# Laboratorio 2. El tablero no cuadra

Alguien dejó un pipeline que dice éxito. Finanzas dice que las ventas no cuadran con el archivo.

Su trabajo no es seguir una receta. Es dejar `oro_fact_ventas` usable y explicar qué apartaron.

Reglas del negocio, nada más:

- Una fila por `venta_id`.
- Solo entran las tiendas 1, 2 y 3.
- `monto` tiene que ser número. Si viene con coma, se interpreta. Si no se puede, se aparta.
- Una columna que no estaba en el contrato no entra al hecho.
- Lo apartado queda en `cuarentena_ventas`, con un motivo.
- Correr el notebook otra vez no duplica el hecho.

No editen el CSV. Si una prueba falla, el hallazgo es de ustedes.

In [0]:
dbutils.widgets.text("ruta_csv", "file:/Workspace/Users/202400878@ujmd.edu.sv/LAB_2/ventas_export.csv")
ruta = dbutils.widgets.get("ruta_csv")

bronce = spark.read.option("header", True).option("inferSchema", False).csv(ruta)
bronce.createOrReplaceTempView("bronce_ventas")

spark.sql("""
create or replace temp view dim_tienda as
select * from values
  (1, 'Centro'),
  (2, 'Escalon'),
  (3, 'Soyapango')
as t(tienda_id, nombre)
""")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)

## El pipeline que heredaron

Esta celda es el flujo roto. Léanla, no la copien como entrega. Carga todo, no aparta nada y si la corren dos veces duplica.

In [0]:
spark.sql("""
create or replace temp view oro_roto as
select * from bronce_ventas
""")
display(spark.sql("select count(*) as filas_oro_roto from oro_roto"))

## Su pipeline

Reemplacen esta celda. Tiene que crear `oro_fact_ventas` y `cuarentena_ventas`.

`oro_fact_ventas`: `venta_id`, `fecha`, `tienda_id`, `producto_id`, `cantidad`, `monto`, `medio_pago`.

`cuarentena_ventas`: las que no entran, con una columna `motivo`.

In [0]:
# Pipeline de ventas

# 1. Normalizar datos y quitar duplicados
spark.sql("""
CREATE OR REPLACE TEMP VIEW ventas_limpias AS

SELECT
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    monto,
    medio_pago,
    canal,

    CASE
        WHEN TRIM(monto) RLIKE '^[0-9]{1,3}(\\.[0-9]{3})+,[0-9]{2}$'
        THEN TRY_CAST(
            REPLACE(
                REPLACE(TRIM(monto), '.', ''),
                ',', '.'
            ) AS DECIMAL(15,2)
        )
        ELSE TRY_CAST(
            REPLACE(TRIM(monto), ',', '.')
            AS DECIMAL(15,2)
        )
    END AS monto_numero,

    ROW_NUMBER() OVER (
        PARTITION BY venta_id
        ORDER BY fecha
    ) AS fila

FROM bronce_ventas
""")

# 2. Tabla Oro
spark.sql("""
CREATE OR REPLACE TEMP VIEW oro_fact_ventas AS

SELECT
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    monto_numero AS monto,
    medio_pago

FROM ventas_limpias

WHERE fila = 1
  AND tienda_id IN (1, 2, 3)
  AND monto_numero IS NOT NULL
""")

# 3. Tabla de cuarentena
spark.sql("""
CREATE OR REPLACE TEMP VIEW cuarentena_ventas AS

SELECT
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    monto,
    medio_pago,
    canal,

    CASE
        WHEN fila > 1
            THEN 'venta_id duplicado'
        WHEN tienda_id IS NULL OR tienda_id NOT IN (1, 2, 3)
            THEN 'tienda no permitida'
        WHEN monto_numero IS NULL
            THEN 'monto inválido'
    END AS motivo

FROM ventas_limpias

WHERE fila > 1
   OR tienda_id IS NULL
   OR tienda_id NOT IN (1, 2, 3)
   OR monto_numero IS NULL
""")

print("Oro:", spark.table("oro_fact_ventas").count())
print("Cuarentena:", spark.table("cuarentena_ventas").count())

## Pruebas

No dicen cómo arreglarlo. Dicen si ya quedó. Las seis tienen que decir PASÓ.

In [0]:
oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
  select venta_id from oro_fact_ventas
  group by venta_id having count(*) > 1
)""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [c for c in oro.columns if c.lower() == "canal"]
sin_motivo = "motivo" not in [c.lower() for c in cuarentena.columns]
oro_inflado = oro.count() >= bronce.count()
cuarentena_vacia = cuarentena.count() == 0

ok = []
ok.append(prueba("una fila por venta_id", duplicados == 0))
ok.append(prueba("solo tiendas 1, 2 y 3", tiendas_malas == 0))
ok.append(prueba("monto numérico en el hecho", montos_malos == 0))
ok.append(prueba("canal no entra al hecho", len(columnas_de_mas) == 0))
ok.append(prueba(
    "cuarentena con motivo y con filas",
    (not sin_motivo) and (not cuarentena_vacia)
))
ok.append(prueba("oro menor que el archivo", not oro_inflado))

print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")

## Desafío o reto

Cuando las seis digan PASÓ, respondan en esta celda, sin mirar la guía:

1. ¿Qué pasó con la misma venta llegada dos veces?
cuando encontré una venta repetida dejé un solo registro en la tabla oro y mandé el registro duplicado a cuarentena para evitar que la venta se contara más de una vez

2. ¿Qué hicieron con un monto que venía en texto?
convertí el monto que venía como texto a un valor numérico para poder trabajarlo correctamente y si no se podía convertir lo envié a cuarentena

3. ¿Por qué una tienda se quedó fuera?
la tienda quedó fuera porque no estaba dentro de las tiendas permitidas para la tabla oro que son la 1 la 2 y la 3

4. ¿Dónde quedó la columna que no estaba en el contrato?
la columna canal quedó en cuarentena porque no formaba parte de las columnas definidas para la tabla oro pero se conservó para no perder esa información

5. Si corren el notebook otra vez, ¿el hecho crece?
no, porque el proceso vuelve a crear las tablas cada vez que se ejecuta entonces los registros no se van acumulando ni se duplican

El que solo pega la consulta y no puede explicar una de estas, no termino el laboratorio